# Prophet — Walmart Store Sales Forecasting (classical baseline)

**Architecture:** Prophet (additive model: trend + yearly seasonality + holidays). It
is a *classical, per-series* method — one model is fit for each time series, unlike the
global neural models that learn from all series at once.

**Why only a sample:** there are ~3,300 series, and Prophet fits a separate model per
series, which is slow. Following the lecturer's guidance not to spend much training
time on classical models, we fit Prophet on a representative **sample of series** and
report the aggregate WMAE as a classical reference point.

**Result:** WMAE 1,529 on a 30-series sample. NOTE: this is *not* directly comparable
to the neural models, whose WMAE is over all ~3,300 series (including sparse, noisy
ones). The sampled series are individually well-behaved, so this is only a reference.

**Evaluation:** WMAE on the validation period (holiday weeks weighted 5x).
**MLflow structure:** experiment `Prophet_Training` with runs `Prophet_Data_Prep`,
`Prophet_Subset`.

In [ ]:
# Run from the project root so data/ and src/ resolve.
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

In [ ]:
import logging
import warnings
import numpy as np
import pandas as pd
import mlflow
import dagshub
from prophet import Prophet

from src.features.nn_preprocessing import build_long_df, temporal_split, FREQ
from src.evaluation.metrics import calculate_wmae

warnings.filterwarnings("ignore")
# Prophet is very chatty; quiet its backend down.
logging.getLogger("cmdstanpy").setLevel(logging.ERROR)
logging.getLogger("prophet").setLevel(logging.ERROR)

# Experiment tracking: DagsHub-hosted MLflow.
dagshub.init(repo_owner="GigiSichinava", repo_name="Walmart-Sales-Forecasting", mlflow=True)
mlflow.set_experiment("Prophet_Training")

SPLIT_DATE = "2012-01-01"
SEED = 42
SAMPLE_SIZE = 30
print("Tracking URI:", mlflow.get_tracking_uri())

In [ ]:
train_raw = pd.read_csv("data/train.csv")
stores = pd.read_csv("data/stores.csv")
features = pd.read_csv("data/features.csv")
print("train:", train_raw.shape)

## Run 1 — Data preparation

In [ ]:
with mlflow.start_run(run_name="Prophet_Data_Prep"):
    Y_df = build_long_df(train_raw, stores, features, fill_gaps=True)
    train_df, valid_df, horizon = temporal_split(Y_df, SPLIT_DATE)

    mlflow.log_param("freq", FREQ)
    mlflow.log_param("split_date", SPLIT_DATE)
    mlflow.log_param("sample_size", SAMPLE_SIZE)
    mlflow.log_param("horizon_weeks", int(horizon))

    print("series:", Y_df["unique_id"].nunique(), "| horizon (weeks):", horizon)

## Run 2 — Prophet on a sample of series

Pick series that have enough history and appear in the validation window, fit one
Prophet per series, forecast the validation horizon, and pool the errors into a single
WMAE.

In [ ]:
# Choose eligible series: enough training weeks and present in validation.
train_counts = train_df.groupby("unique_id").size()
valid_ids = set(valid_df["unique_id"].unique())
eligible = [uid for uid in train_counts.index if train_counts[uid] >= 60 and uid in valid_ids]

rng = np.random.default_rng(SEED)
sample_ids = list(rng.choice(eligible, size=min(SAMPLE_SIZE, len(eligible)), replace=False))
print("eligible series:", len(eligible), "| sampled:", len(sample_ids))

In [ ]:
with mlflow.start_run(run_name="Prophet_Subset"):
    mlflow.log_param("sample_size", len(sample_ids))
    mlflow.log_param("yearly_seasonality", True)

    all_true, all_pred, all_hol = [], [], []
    for i, uid in enumerate(sample_ids, 1):
        tr = train_df[train_df["unique_id"] == uid][["ds", "y"]].sort_values("ds")
        m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
        m.fit(tr)

        future = m.make_future_dataframe(periods=horizon, freq=FREQ)
        fc = m.predict(future)[["ds", "yhat"]]

        va = valid_df[valid_df["unique_id"] == uid][["ds", "y", "IsHoliday"]]
        merged = va.merge(fc, on="ds", how="inner")
        all_true.extend(merged["y"].tolist())
        all_pred.extend(merged["yhat"].clip(lower=0).tolist())
        all_hol.extend(merged["IsHoliday"].tolist())
        if i % 10 == 0:
            print(f"  fitted {i}/{len(sample_ids)} series")

    wmae = calculate_wmae(np.array(all_true), np.array(all_pred), np.array(all_hol))
    mlflow.log_metric("WMAE", float(wmae))
    print(f"Prophet subset WMAE ({len(sample_ids)} series): {wmae:,.2f}")

## Notes

- Prophet subset WMAE: 1,529 (30 series). This is on a sample, not all series, so it is
  a rough reference and NOT directly comparable to the global neural models' full-set
  WMAE.
- Key contrast for the report: classical models fit one model per series and cannot
  share information across series, unlike the global neural forecasters. They also do
  not scale well to thousands of series.